# Calculate annual PM2.5 mean

In [ ]:
import os
import xarray as xr
from utils.utils import get_scenario_config

In [ ]:
# === Processing Function ===
def calculate_annual_surface_pm25(pm25_surf):
    # Compute annual mean
    print("Calculating annual mean")
    annual_mean = pm25_surf.groupby("time.year").mean("time")
    return annual_mean

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

PM25_DIR = f"/glade/work/awells/air_quality/{model}/pm25/monthly_pm25/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25/"

# === Main Loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")

    dates = f"{years.start}01-{years.stop}12"

    file = f"Monthly_PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    file_path = os.path.join(PM25_DIR, file)
    da = xr.open_dataarray(file_path)

    annual_pm25 = calculate_annual_surface_pm25(da)

    new_dates = f"{years.start}-{years.stop}"

    out_file = f"Annual_PM25_{model}_{scenario}_{ens_num:02d}_{new_dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    description = ("Annual mean PM2.5 - scripts by A.F. Wells (2025)")
    annual_pm25.attrs["description"] = description
    annual_pm25.attrs["ensemble_number"] = ens_num
    annual_pm25.attrs["scenario"] = scenario
    annual_pm25.attrs["model"] = model

    print(f"Saving annual PM2.5 to {out_path}")
    annual_pm25.to_netcdf(out_path)

print("All processing complete.")